In [ ]:
#r "nuget: RProvider,{{package-version}}"


# Quickstart: Using Statistical Packages

A strong R community has contributed over 20,000 packages to CRAN,
R's central package registry. The F# R Type Provider enables you to
use every single one of them from within the F# environment.

Using RRrovider, you can orchestrate R workflows and manipulate R data,
pass in F# values, and extract R values back to F#.

For this example, we simply demonstrate some basic RProvider concepts
using the built-in `stats` package.

## Example: Linear Regression

Let's perform a simple linear regression from the F# interactive,
using the R.lm function.

Once you have referenced RProvider's nuget package in your script,
library, or app, you can reference the required libraries and packages this way:



In [2]:
open RProvider
open RProvider.Operators

open RProvider.graphics
open RProvider.stats


Once the libraries and packages have been loaded,
Imagine that our true model is

Y = 5.0 + 3.0 * X1 - 2.0 * X2 + noise

Let's generate a fake dataset using F# that follows this model:



In [3]:
// Random number generator
let rng = System.Random()
let rand () = rng.NextDouble()

// Generate fake X1 and X2 
let X1s = [ for i in 0 .. 9 -> 10. * rand () ]
let X2s = [ for i in 0 .. 9 -> 5. * rand () ]

// Build Ys, following the "true" model
let Ys = [ for i in 0 .. 9 -> 5. + 3. * X1s.[i] - 2. * X2s.[i] + rand () ]


Using linear regression on this dataset, we should be able to
estimate the coefficients 5.0, 3.0 and -2.0, with some imprecision
due to the "noise" part.

Let's first put our dataset into a R dataframe; this allows us
to name our vectors, and use these names in R formulas afterwards:



In [4]:
let dataset = [ 
    "Y" => Ys
    "X1" => X1s
    "X2" => X2s ] |> R.data_frame


We can now use R to perform a linear regression.
We call the [R.lm function](http://stat.ethz.ch/R-manual/R-patched/library/stats/html/lm.html),
passing it the formula we want to estimate.
(See the [R manual on formulas](http://stat.ethz.ch/R-manual/R-patched/library/stats/html/formula.html)
for more on their somewhat esoteric construction)



In [5]:
let result = R.lm(formula = "Y~X1+X2", data = dataset)


## Extracting Results from R to F#

The result we get back from R is a R Expression.
The R Type Provider tries as much as possible to keep data
as R Expressions, rather than converting back-and-forth
between F# and R types. It limits translations
between the 2 languages, which has performance benefits,
and simplifies composing R operations. On the other hand,
we need to extract the results from the R expression
into F# types.

The [R docs for lm](http://stat.ethz.ch/R-manual/R-patched/library/stats/html/lm.html)
describes what R.lm returns: a R List. We can now retrieve each element,
accessing it by name (as defined in the documentation).
For instance, let's retrieve the coefficients and residuals,
which are both R vectors containg floats:



In [6]:
let coefficients = result?coefficients.AsVector().AsReal()
let residuals = result?residuals.AsVector().AsReal()


We can also produce summary statistics about our model,
like R^2, which measures goodness-of-fit - close to 0
indicates a very poor fit, and close to 1 a good fit.
See [R docs for the details on Summary](http://stat.ethz.ch/R-manual/R-patched/library/stats/html/summary.lm.html).



In [7]:
let summary = R.summary result

summary?``r.squared``.AsScalar()


NumericS { Sexp = { ptr = 4627474096n } }

Finally, we can directly pass results, which is a R expression,
to R.plot, to produce some fancy charts describing our model:



In [8]:
Graphics.svg 8 4 (fun _ -> R.plot result)


<?xml version='1.0' encoding='UTF-8' ?> 0.0 0.1 0.2 0.3 0.4 0.5 -1.5 -0.5 0.5 1.5 Leverage Standardized residuals (function (formula, data, subset, weights, na.action, method = "qr", model ... <polyline points='81.57,-595.15 86.22,-373.44 90.86,-276.35 95.50,-218.64 100.14,-179.26 104.78,-150.16 109.42,-127.52 114.07,-109.23 118.71,-94.05 123.35,-81.19 127.99,-70.09 132.63,-60.39 137.28,-51.81 141.92,-44.14 146.56,-37.24 151.20,-30.98 155.84,-25.26 160.49,-20.02 165.13,-15.18 169.77,-10.69 174.41,-6.52 179.05,-2.63 183.69,1.02 188.34,4.45 192.98,7.68 197.62,10.73 202.26,13.62 206.90,16.36 211.55,18.96 216.19,21.44 220.83,23.81 225.47,26.07 230.11,28.24 234.75,30.31 239.40,32.30 244.04,34.22 248.68,36.06 253.32,37.84 257.96,39.55 262.61,41.20 267.25,42.80 271.89,44.34 276.53,45.84 281.17,47.28 285.82,48.69 290.46,50.05 295.10,51.38 299.74,52.66 304.38,53.91 309.02,55.13 313.67,56.32 318.31,57.47 322.95,58.60 327.59,59.70 332.23,60.77 336.88,61.82 341.52,62.85 346.16,63.85 350.80,64.83 355.44,65.79 360.09,66.73 364.73,67.64 369.37,68.55 374.01,69.43 378.65,70.29 383.29,71.14 387.94,71.98 392.58,72.79 397.22,73.60 401.86,74.39 406.50,75.16 411.15,75.93 415.79,76.68 420.43,77.42 425.07,78.14 429.71,78.86 434.36,79.56 439.00,80.26 443.64,80.94 448.28,81.61 452.92,82.28 457.56,82.93 462.21,83.58 466.85,84.21 471.49,84.84 476.13,85.46 480.77,86.08 485.42,86.68 490.06,87.28 494.70,87.87 499.34,88.45 503.98,89.03 508.63,89.60 513.27,90.17 517.91,90.72 522.55,91.28 527.19,91.82 531.83,92.37 536.48,92.90 541.12,93.43 545.76,93.96 ' style='stroke-width: 0.75; stroke: #9E9E9E; stroke-dasharray: 4.00,4.00;' /><polyline points='81.57,883.02 86.22,661.31 90.86,564.22 95.50,506.50 100.14,467.12 104.78,438.03 109.42,415.38 114.07,397.10 118.71,381.92 123.35,369.05 127.99,357.95 132.63,348.25 137.28,339.67 141.92,332.01 146.56,325.10 151.20,318.84 155.84,313.13 160.49,307.88 165.13,303.04 169.77,298.56 174.41,294.39 179.05,290.49 183.69,286.84 188.34,283.41 192.98,280.19 197.62,277.13 202.26,274.25 206.90,271.51 211.55,268.90 216.19,266.42 220.83,264.05 225.47,261.79 230.11,259.63 234.75,257.55 239.40,255.56 244.04,253.65 248.68,251.80 253.32,250.03 257.96,248.32 262.61,246.67 267.25,245.07 271.89,243.52 276.53,242.03 281.17,240.58 285.82,239.18 290.46,237.81 295.10,236.49 299.74,235.20 304.38,233.95 309.02,232.73 313.67,231.55 318.31,230.39 322.95,229.26 327.59,228.16 332.23,227.09 336.88,226.04 341.52,225.02 346.16,224.02 350.80,223.04 355.44,222.08 360.09,221.14 364.73,220.22 369.37,219.32 374.01,218.44 378.65,217.57 383.29,216.72 387.94,215.89 392.58,215.07 397.22,214.27 401.86,213.48 406.50,212.70 411.15,211.94 415.79,211.19 420.43,210.45 425.07,209.72 429.71,209.01 434.36,208.30 439.00,207.61 443.64,206.93 448.28,206.25 452.92,205.59 457.56,204.93 462.21,204.29 466.85,203.65 471.49,203.02 476.13,202.40 480.77,201.79 485.42,201.18 490.06,200.59 494.70,199.99 499.34,199.41 503.98,198.83 508.63,198.26 513.27,197.70 517.91,197.14 522.55,196.59 527.19,196.04 531.83,195.50 536.48,194.96 541.12,194.43 545.76,193.91 ' style='stroke-width: 0.75; stroke: #9E9E9E; stroke-dasharray: 4.00,4.00;' /><polyline points='81.57,-901.29 86.22,-587.75 90.86,-450.44 95.50,-368.82 100.14,-313.13 104.78,-271.98 109.42,-239.96 114.07,-214.10 118.71,-192.63 123.35,-174.43 127.99,-158.74 132.63,-145.02 137.28,-132.88 141.92,-122.05 146.56,-112.28 151.20,-103.43 155.84,-95.35 160.49,-87.93 165.13,-81.08 169.77,-74.74 174.41,-68.84 179.05,-63.33 183.69,-58.17 188.34,-53.33 192.98,-48.76 197.62,-44.44 202.26,-40.36 206.90,-36.49 211.55,-32.80 216.19,-29.29 220.83,-25.95 225.47,-22.75 230.11,-19.69 234.75,-16.75 239.40,-13.93 244.04,-11.23 248.68,-8.62 253.32,-6.11 257.96,-3.69 262.61,-1.35 267.25,0.90 271.89,3.09 276.53,5.20 281.17,7.25 285.82,9.24 290.46,11.17 295.10,13.04 299.74,14.86 304.38,16.63 309.02,18.35 313.67,20.03 318.31,21.66 322.95,23.26 327.59,24.81 332.23,26.33 336.88,27.81 341.52,29.26 346.16,30.68 350.80,32.06 355.44,33.42 360.09,34.75 364.73,36.05 369.

That's it - while simple, we hope this example illustrate
how you would go about to use any existing R statistical package.
While the details would differ, the general approach would
remain the same. Happy modelling!

